# Butterfly Inception Localization 
object localization (bbox) dengan InceptionV3 + Grad-CAM.
Model dilatih dengan 2 tahap (head training + fine-tuning), lalu dipakai untuk membuat output bbox pada data test.

## 1. Setup

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input as inception_preprocess

from sklearn.model_selection import train_test_split

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)
AUTOTUNE = tf.data.AUTOTUNE
IMG_SIZE = 299

## 2. Konfigurasi Path dan Hyperparameter

In [ ]:
DATA_DIR = Path('/kaggle/input/datasets/phucthaiv02/butterfly-image-classification')

TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
TRAIN_CSV = DATA_DIR / 'Training_set.csv'
TEST_CSV = DATA_DIR / 'Testing_set.csv'

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUTPUT_DIR = WORK_DIR / 'butterfly_localization_only'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEACHER_MODEL_OUT = OUTPUT_DIR / 'teacher_inception_classifier.keras'
PRED_TEST_CSV = OUTPUT_DIR / 'test_localization_only.csv'

VAL_SIZE = 0.2
BATCH_SIZE = 32
EPOCHS_TEACHER_HEAD = 4
EPOCHS_TEACHER_FINETUNE = 6
LR_TEACHER_HEAD = 1e-3
LR_TEACHER_FINETUNE = 1e-5

assert TRAIN_DIR.exists(), f'Missing dir: {TRAIN_DIR}'
assert TEST_DIR.exists(), f'Missing dir: {TEST_DIR}'
assert TRAIN_CSV.exists(), f'Missing file: {TRAIN_CSV}'
assert TEST_CSV.exists(), f'Missing file: {TEST_CSV}'

{
    'data_dir': str(DATA_DIR),
    'output_dir': str(OUTPUT_DIR),
}

## 3. Load Metadata dan Split Train/Val

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

class_names = sorted(train_df['label'].unique())
class_to_idx = {name: i for i, name in enumerate(class_names)}
idx_to_class = dict(enumerate(class_names))
num_classes = len(class_names)

train_df = train_df.assign(
    label_idx=train_df['label'].map(class_to_idx).astype('int32')
)

can_stratify = train_df['label_idx'].value_counts().min() >= 2
tr_df, val_df = train_test_split(
    train_df,
    test_size=VAL_SIZE,
    random_state=42,
    stratify=train_df['label_idx'] if can_stratify else None,
 )

tr_df = tr_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

{
    'num_classes': num_classes,
    'train_size': len(tr_df),
    'val_size': len(val_df),
    'test_size': len(test_df),
}

## 4. Train Teacher (Head + Fine-Tuning) + Grad-CAM

In [ ]:
def make_cls_arrays(df, image_dir):
    return (
        (image_dir / df['filename']).astype(str).values,
        df['label_idx'].astype('int32').values,
    )

def parse_cls(path, label, training=False):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)
    if training:
        img = tf.image.random_flip_left_right(img)
    return inception_preprocess(img), label

def build_cls_ds(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(paths), 2048), reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: parse_cls(p, y, training=training), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

tr_paths, tr_labels = make_cls_arrays(tr_df, TRAIN_DIR)
val_paths, val_labels = make_cls_arrays(val_df, TRAIN_DIR)
teacher_train_ds = build_cls_ds(tr_paths, tr_labels, training=True)
teacher_val_ds = build_cls_ds(val_paths, val_labels, training=False)

teacher_backbone = InceptionV3(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
teacher_backbone.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = teacher_backbone(inputs, training=False)  # Keep BN in inference mode during fine-tune.
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
teacher_model = keras.Model(inputs, outputs, name='teacher_inception_classifier')

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(TEACHER_MODEL_OUT),
        monitor='val_acc',
        save_best_only=True,
        mode='max',
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_acc',
        patience=3,
        restore_best_weights=True,
        mode='max',
        verbose=1,
    ),
]

def compile_teacher(lr):
    teacher_model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name='acc')],
    )

# Stage 1: train head.
compile_teacher(LR_TEACHER_HEAD)
teacher_model.fit(
    teacher_train_ds,
    validation_data=teacher_val_ds,
    epochs=EPOCHS_TEACHER_HEAD,
    callbacks=callbacks,
    verbose=1,
)

# Stage 2: fine-tune last blocks with low learning rate.
teacher_backbone.trainable = True
for layer in teacher_backbone.layers[:-40]:
    layer.trainable = False

compile_teacher(LR_TEACHER_FINETUNE)
teacher_model.fit(
    teacher_train_ds,
    validation_data=teacher_val_ds,
    epochs=EPOCHS_TEACHER_FINETUNE,
    callbacks=callbacks,
    verbose=1,
)

teacher_model = keras.models.load_model(TEACHER_MODEL_OUT)

def find_last_4d_layer_name(model):
    for layer in reversed(model.layers):
        shape = getattr(layer, 'output_shape', None)
        if isinstance(shape, tuple) and len(shape) == 4:
            return layer.name
    return None

teacher_target_layer = find_last_4d_layer_name(teacher_model)
assert teacher_target_layer is not None, 'Tidak menemukan layer 4D untuk Grad-CAM teacher model.'

def forward_safe(model, batch):
    x = tf.convert_to_tensor(batch, dtype=tf.float32)
    last_err = None
    for payload in (x, (x,), [x]):
        try:
            return model(payload, training=False)
        except Exception as e:
            last_err = e
    raise RuntimeError(f'forward_safe gagal untuk semua format input: {last_err}')

def load_inception_batch(image_path: Path):
    img = keras.utils.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = keras.utils.img_to_array(img).astype('float32')
    return arr, inception_preprocess(np.expand_dims(arr, axis=0))

def make_gradcam_heatmap(img_batch, model, target_layer_name, pred_index):
    grad_model = keras.Model(
        model.inputs, [model.get_layer(target_layer_name).output, model.output]
    )
    x = tf.convert_to_tensor(img_batch, dtype=tf.float32)
    with tf.GradientTape() as tape:
        try:
            fmap, preds = grad_model(x, training=False)
        except Exception:
            fmap, preds = forward_safe(grad_model, x)
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, fmap)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.squeeze(fmap[0] @ weights[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0)
    return (heatmap / (tf.reduce_max(heatmap) + 1e-9)).numpy()

def bbox_from_heatmap(heatmap, out_w, out_h):
    hm = np.nan_to_num(heatmap, nan=0.0, posinf=0.0, neginf=0.0).astype('float32')
    hm = hm / (hm.max() + 1e-9)
    mask = hm >= max(0.2, float(np.quantile(hm, 0.85)))
    ys, xs = np.where(mask)
    if xs.size == 0:
        return 0, 0, out_w - 1, out_h - 1

    x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
    x1 = int(np.clip(x1, 0, out_w - 1))
    y1 = int(np.clip(y1, 0, out_h - 1))
    x2 = int(np.clip(x2, 0, out_w - 1))
    y2 = int(np.clip(y2, 0, out_h - 1))
    if x2 <= x1 or y2 <= y1:
        return 0, 0, out_w - 1, out_h - 1
    return x1, y1, x2, y2

def to_norm_xyxy(x1, y1, x2, y2, w, h):
    return [
        np.clip(x1 / w, 0.0, 1.0),
        np.clip(y1 / h, 0.0, 1.0),
        np.clip(x2 / w, 0.0, 1.0),
        np.clip(y2 / h, 0.0, 1.0),
    ]

{
    'teacher_model_out': str(TEACHER_MODEL_OUT),
    'teacher_target_layer': teacher_target_layer,
}

## 5. Prediksi Test Localization dan Simpan Output

In [ ]:
def _call_no_dict(model, x):
    for payload in (x, (x,), [x]):
        try:
            return model(payload, training=False)
        except Exception:
            continue
    raise RuntimeError('Model call gagal untuk payload tensor/tuple/list.')

def make_cam_heatmap_local(img_batch, model, pred_index):
    x = tf.convert_to_tensor(img_batch, dtype=tf.float32)
    backbone = model.get_layer('inception_v3')
    fmap = _call_no_dict(backbone, x)  # (1, H, W, C)

    dense = model.layers[-1]
    kernel = dense.get_weights()[0]  # (C, num_classes)
    class_w = tf.convert_to_tensor(kernel[:, pred_index], dtype=tf.float32)

    heatmap = tf.tensordot(fmap[0], class_w, axes=([2], [0]))
    heatmap = tf.maximum(heatmap, 0)
    return (heatmap / (tf.reduce_max(heatmap) + 1e-9)).numpy()

def build_localization_df(df, image_dir):
    rows = []
    for _, r in df.iterrows():
        image_path = image_dir / r['filename']
        if not image_path.exists():
            continue

        pil = keras.utils.load_img(image_path)
        w, h = pil.size
        _, batch = load_inception_batch(image_path)

        probs = teacher_model.predict(batch, verbose=0)[0]
        pred_idx = int(np.argmax(probs))
        conf = float(probs[pred_idx])

        heatmap = make_cam_heatmap_local(batch, teacher_model, pred_idx)
        heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (h, w), method='bilinear').numpy().squeeze()
        x1, y1, x2, y2 = bbox_from_heatmap(heatmap_resized, w, h)
        x1n, y1n, x2n, y2n = to_norm_xyxy(x1, y1, x2, y2, w, h)

        rows.append({
            'filename': r['filename'],
            'label': idx_to_class[pred_idx],
            'confidence': conf,
            'x1': x1n,
            'y1': y1n,
            'x2': x2n,
            'y2': y2n,
        })
    return pd.DataFrame(rows)

pred_df = build_localization_df(test_df, TEST_DIR)
assert len(pred_df) > 0, 'Localization output kosong.'
pred_df.to_csv(PRED_TEST_CSV, index=False)

run_meta = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'teacher_model_path': str(TEACHER_MODEL_OUT),
    'test_prediction_csv': str(PRED_TEST_CSV),
    'num_classes': int(num_classes),
    'test_rows': int(len(pred_df)),
    'localization_method': 'cam_gap_dense',
}

with (OUTPUT_DIR / 'run_metadata.json').open('w') as f:
    json.dump(run_meta, f, indent=2)

PRED_TEST_CSV, pred_df.head()

In [ ]:
# Visual sanity check sederhana.
sample_df = pred_df.sample(n=min(5, len(pred_df)), random_state=42).reset_index(drop=True)
fig, axes = plt.subplots(len(sample_df), 1, figsize=(10, 4 * len(sample_df)))
if len(sample_df) == 1:
    axes = [axes]

for i, row in sample_df.iterrows():
    img_path = TEST_DIR / row['filename']
    img = keras.utils.load_img(img_path)
    arr = keras.utils.img_to_array(img).astype('uint8')
    h, w = arr.shape[:2]

    x1 = int(np.clip(row['x1'], 0, 1) * w)
    y1 = int(np.clip(row['y1'], 0, 1) * h)
    x2 = int(np.clip(row['x2'], 0, 1) * w)
    y2 = int(np.clip(row['y2'], 0, 1) * h)

    x1, x2 = min(x1, x2), max(x1, x2)
    y1, y2 = min(y1, y2), max(y1, y2)
    bw, bh = max(1, x2 - x1), max(1, y2 - y1)

    axes[i].imshow(arr)
    axes[i].add_patch(plt.Rectangle((x1, y1), bw, bh, fill=False, color='red', linewidth=2))
    axes[i].set_title(f"{row['filename']} | {row['label']} ({row['confidence']:.3f})")
    axes[i].axis('off')

plt.tight_layout()
plt.show()